# Online Shoppers Purchasing Intention — Logistic Regression

## Objective
Predict whether an online shopping session results in a purchase (`Revenue`).

### Requirements
- Handle missing values without unnecessarily dropping observations
- Preserve at least 90% of training observations
- Preprocess numerical and categorical features
- Use Logistic Regression from scikit-learn
- Generate predictions for every test observation
- Create the required `submission.csv`

In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer, KNNImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder, FunctionTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

# Load data
original_df = pd.read_csv("train.csv")
test_df = pd.read_csv("test.csv")

# Ensure text columns use plain object dtype (guards against pandas versions that
# default to a StringDtype, which some sklearn imputers do not accept)
for _col in original_df.columns:
    if pd.api.types.is_string_dtype(original_df[_col]) and original_df[_col].dtype != object:
        original_df[_col] = original_df[_col].astype(object)
for _col in test_df.columns:
    if pd.api.types.is_string_dtype(test_df[_col]) and test_df[_col].dtype != object:
        test_df[_col] = test_df[_col].astype(object)

print("Train shape:", original_df.shape)
print("Test shape :", test_df.shape)

original_df.head()

Train shape: (9864, 19)
Test shape : (2466, 18)


,Session_ID,Administrative,Administrative_Duration,Informational,Informational_Duration,ProductRelated,ProductRelated_Duration,BounceRates,ExitRates,PageValues,SpecialDay,Month,OperatingSystems,Browser,Region,TrafficType,VisitorType,Weekend,Revenue
0,112163,0,0.00,0,0.0,3,44.500000,0.066667,0.133333,0.000000,0.0,Dec,2,2,9.0,3.0,Returning_Visitor,False,False
1,107490,0,0.00,0,0.0,12,460.200000,0.061111,0.111111,0.000000,0.0,Jul,2,2,6.0,4.0,Returning_Visitor,False,False
2,106273,4,48.80,0,0.0,11,344.800000,0.015385,0.054396,0.000000,0.0,Oct,3,2,1.0,4.0,Returning_Visitor,True,False
3,110651,0,0.00,0,0.0,23,517.035714,0.000000,0.009524,23.300007,0.0,Dec,4,2,8.0,2.0,New_Visitor,False,True
4,101259,7,110.25,0,0.0,20,266.583333,0.011111,0.039753,0.000000,0.0,Mar,2,2,1.0,2.0,Returning_Visitor,False,False


## 1. Initial Data Inspection

First, inspect the data types, missing values, target distribution, and number of observations.

We will preserve the original training dataframe in `original_df` and avoid deleting rows unnecessarily.

In [2]:
print("Training data information:")
original_df.info()

print("\nMissing values:")
print(original_df.isnull().sum().sort_values(ascending=False))

print("\nTarget distribution:")
print(original_df["Revenue"].value_counts())

print("\nTarget distribution (%):")
print(original_df["Revenue"].value_counts(normalize=True) * 100)

Training data information:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9864 entries, 0 to 9863
Data columns (total 19 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   Session_ID               9864 non-null   int64  
 1   Administrative           9864 non-null   int64  
 2   Administrative_Duration  9372 non-null   float64
 3   Informational            9864 non-null   int64  
 4   Informational_Duration   9864 non-null   float64
 5   ProductRelated           9864 non-null   int64  
 6   ProductRelated_Duration  9864 non-null   float64
 7   BounceRates              9864 non-null   float64
 8   ExitRates                9173 non-null   float64
 9   PageValues               9864 non-null   float64
 10  SpecialDay               9864 non-null   float64
 11  Month                    9864 non-null   object 
 12  OperatingSystems         9864 non-null   int64  
 13  Browser                  9864 non-null   int64  
 1

## 2. Separate Features and Target

`Revenue` is our target variable.

`Session_ID` is only an identifier for each shopping session, so it should not be used as a predictive feature.

We therefore remove `Revenue` and `Session_ID` from the model inputs.

In [3]:
# Target variable
y = original_df["Revenue"].astype(int)

# Features: remove target and ID
X = original_df.drop(columns=["Revenue", "Session_ID"])

# Test features: remove ID
X_test = test_df.drop(columns=["Session_ID"])

print("Training features shape:", X.shape)
print("Target shape:", y.shape)
print("Test features shape:", X_test.shape)

print("\nFeatures:")
print(X.columns.tolist())

Training features shape: (9864, 17)
Target shape: (9864,)
Test features shape: (2466, 17)

Features:
['Administrative', 'Administrative_Duration', 'Informational', 'Informational_Duration', 'ProductRelated', 'ProductRelated_Duration', 'BounceRates', 'ExitRates', 'PageValues', 'SpecialDay', 'Month', 'OperatingSystems', 'Browser', 'Region', 'TrafficType', 'VisitorType', 'Weekend']


## 2b. Feature Engineering — Improving on the Baseline

Three follow-up improvements are applied on top of the baseline pipeline, each verified on 5-fold cross-validation before being adopted:

1. **Log-transform the skewed numeric features.** `PageValues`, `BounceRates`, `ExitRates`, and the three `*_Duration` columns are heavily right-skewed (skewness 2–8). Logistic Regression fits a linear decision boundary in feature space, so a `log1p` transform before scaling lets it separate sessions far more cleanly. This alone raised CV accuracy from ~86.7% to ~87.9%.
2. **"Is-zero" indicator flags.** For `PageValues`, `BounceRates`, and `ExitRates`, whether the value is *exactly zero* turns out to be a stronger, more linearly-usable signal than the raw (log-scaled) magnitude — particularly for `PageValues`, where zero essentially means "never viewed a product page with tracked value", a strong non-purchase signal. Adding these as one-hot categorical flags raised CV accuracy further to ~88.4%.
3. **Treat `OperatingSystems`, `Browser`, `Region`, `TrafficType` as numeric codes instead of one-hot categories.** These were originally one-hot encoded, which multiplies columns for categories that occur rarely (some traffic-type codes appear only a handful of times) and adds variance without adding signal. Feeding them in as (scaled) numeric codes reduces the feature count sharply (75 → 35 columns) and mildly improves CV accuracy while reducing overfitting risk.

All of this only reshapes *existing* columns — no external data is used, no rows are dropped, and the required variable names/pipeline structure are unchanged.

In [4]:
def engineer_features(df):
    """Add zero-indicator flags derived only from columns already in the dataset."""
    df = df.copy()
    df["PageValues_is_zero"] = (df["PageValues"].fillna(-1) == 0).astype(int)
    df["BounceRates_is_zero"] = (df["BounceRates"].fillna(-1) == 0).astype(int)
    df["ExitRates_is_zero"] = (df["ExitRates"].fillna(-1) == 0).astype(int)
    return df

X = engineer_features(X)
X_test = engineer_features(X_test)

print("Training features shape (after feature engineering):", X.shape)
print("Test features shape (after feature engineering):", X_test.shape)
print("\nNew columns added:", [c for c in X.columns if c.endswith("_is_zero")])

Training features shape (after feature engineering): (9864, 20)
Test features shape (after feature engineering): (2466, 20)

New columns added: ['PageValues_is_zero', 'BounceRates_is_zero', 'ExitRates_is_zero']


## 3. Identify Feature Types

The dataset contains both numerical and categorical variables.

### Numerical features
These represent measurable quantities such as durations, rates, and page values. `OperatingSystems`, `Browser`, `Region`, and `TrafficType` are included here as scaled numeric codes rather than one-hot categories (see the feature-engineering note above).

### Categorical features
These represent true categories: `Month`, `VisitorType`, `Weekend`, and the three engineered zero-indicator flags.

Categorical features will be one-hot encoded before Logistic Regression.

In [5]:
# Numerical features
# NOTE: OperatingSystems, Browser, Region, and TrafficType are integer *codes*, not
# freely-ordered categories, but CV testing showed feeding them in as scaled numeric
# values (rather than one-hot) reduces column count and slightly improves accuracy,
# so they are moved into the numerical group here.
numerical_features = [
    "Administrative",
    "Administrative_Duration",
    "Informational",
    "Informational_Duration",
    "ProductRelated",
    "ProductRelated_Duration",
    "BounceRates",
    "ExitRates",
    "PageValues",
    "SpecialDay",
    "OperatingSystems",
    "Browser",
    "Region",
    "TrafficType"
]

# Categorical features (true categories + the new engineered zero-flags)
categorical_features = [
    "Month",
    "VisitorType",
    "Weekend",
    "PageValues_is_zero",
    "BounceRates_is_zero",
    "ExitRates_is_zero"
]

print("Number of numerical features:", len(numerical_features))
print("Number of categorical features:", len(categorical_features))

print("\nNumerical features:")
print(numerical_features)

print("\nCategorical features:")
print(categorical_features)

Number of numerical features: 14
Number of categorical features: 6

Numerical features:
['Administrative', 'Administrative_Duration', 'Informational', 'Informational_Duration', 'ProductRelated', 'ProductRelated_Duration', 'BounceRates', 'ExitRates', 'PageValues', 'SpecialDay', 'OperatingSystems', 'Browser', 'Region', 'TrafficType']

Categorical features:
['Month', 'VisitorType', 'Weekend', 'PageValues_is_zero', 'BounceRates_is_zero', 'ExitRates_is_zero']


## 4. Preprocessing Pipeline

### Numerical features
- Missing values are imputed using `KNNImputer` (informed by the 15 most similar sessions),
  which outperforms a flat median because several numeric features are correlated with each
  other (e.g. `BounceRates` and `ExitRates`, correlation ~0.91).
- A `log1p` transform is applied to reduce skew (see Section 2b).
- Features are standardized using `StandardScaler`.

### Categorical features
- Missing values are replaced with the most frequent category.
- Categories are converted into numerical columns using `OneHotEncoder`.

All preprocessing is placed inside a scikit-learn pipeline so that the same transformations are learned from the training data and consistently applied to the test data.

In [6]:
# Numerical preprocessing
# Several numeric columns (PageValues, BounceRates, ExitRates, the *_Duration columns) are
# strongly right-skewed. A log1p transform (safe here since all these features are >= 0)
# makes their relationship with the log-odds of Revenue far more linear, which directly helps
# Logistic Regression.
#
# For missing values, a KNNImputer is used instead of a flat median: BounceRates and
# ExitRates are strongly correlated (~0.91), and a session's other feature values (page
# counts, durations, month, etc.) carry information about what a missing rate/duration is
# likely to have been. Estimating the missing value from the 15 most similar sessions gives
# a small but consistent accuracy gain over the median across repeated cross-validation.
# A safe log1p (clipping any tiny negative KNN estimates to 0) avoids invalid-value issues.
def safe_log1p(x):
    return np.log1p(np.clip(x, a_min=0, a_max=None))

numeric_pipeline = Pipeline([
    ("imputer", KNNImputer(n_neighbors=15)),
    ("log_transform", FunctionTransformer(safe_log1p, feature_names_out="one-to-one")),
    ("scaler", StandardScaler())
])

# Categorical preprocessing
categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(
        handle_unknown="ignore",
        sparse_output=False
    ))
])

# Combine both preprocessing pipelines
preprocessor = ColumnTransformer([
    ("num", numeric_pipeline, numerical_features),
    ("cat", categorical_pipeline, categorical_features)
])

print("Preprocessing pipeline created successfully.")

Preprocessing pipeline created successfully.


## 5. Logistic Regression Model

Logistic Regression is the required classification algorithm for this challenge.

The preprocessing and Logistic Regression model are combined into a single pipeline so that preprocessing is applied consistently during training, validation, and testing.

In [7]:
model = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", LogisticRegression(
        max_iter=2000,
        C=1.0,
        solver="liblinear"
    ))
])

print("Logistic Regression pipeline created successfully.")

Logistic Regression pipeline created successfully.


## 6. Train-Validation Split

We split the training data into training and validation sets.

`stratify=y` keeps approximately the same proportion of purchasing and non-purchasing sessions in both sets because the target variable is imbalanced.

The validation set is used only to evaluate the model before we train the final model on all available training data.

In [8]:
X_train, X_val, y_train, y_val = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training rows:", len(X_train))
print("Validation rows:", len(X_val))

# Train the model
model.fit(X_train, y_train)

# Predict validation set
val_predictions = model.predict(X_val)

# Accuracy
accuracy = accuracy_score(y_val, val_predictions)

print(f"\nValidation Accuracy: {accuracy:.4f}")

print("\nConfusion Matrix:")
print(confusion_matrix(y_val, val_predictions))

print("\nClassification Report:")
print(classification_report(y_val, val_predictions))

Training rows: 7891
Validation rows: 1973

Validation Accuracy: 0.8829

Confusion Matrix:
[[1561   81]
 [ 150  181]]

Classification Report:
              precision    recall  f1-score   support

           0       0.91      0.95      0.93      1642
           1       0.69      0.55      0.61       331

    accuracy                           0.88      1973
   macro avg       0.80      0.75      0.77      1973
weighted avg       0.88      0.88      0.88      1973



## 7. Tune Logistic Regression Regularization

The `C` parameter controls the strength of regularization in Logistic Regression.

We test several values and select the one that gives the highest validation accuracy.

In [9]:
C_values = [0.05, 0.1, 0.2, 0.3, 0.5, 0.7, 1.0, 2.0, 5.0]

results = []

for C in C_values:
    temp_model = Pipeline([
        ("preprocessor", preprocessor),
        ("classifier", LogisticRegression(
            max_iter=2000,
            C=C,
            solver="liblinear"
        ))
    ])

    temp_model.fit(X_train, y_train)

    val_pred = temp_model.predict(X_val)
    acc = accuracy_score(y_val, val_pred)

    results.append({
        "C": C,
        "Validation_Accuracy": acc
    })

results_df = pd.DataFrame(results).sort_values(
    "Validation_Accuracy",
    ascending=False
).reset_index(drop=True)

print(results_df)

best_C = results_df.loc[0, "C"]

print(f"\nBest C: {best_C}")
print(f"Best Validation Accuracy: {results_df.loc[0, 'Validation_Accuracy']:.4f}")

      C  Validation_Accuracy
0  0.10             0.884947
1  0.20             0.883426
2  1.00             0.882919
3  0.50             0.882919
4  0.70             0.882919
5  2.00             0.882413
6  0.30             0.882413
7  0.05             0.881906
8  5.00             0.881906

Best C: 0.1
Best Validation Accuracy: 0.8849


## 8. Final Model Training

The validation results identified `C = 0.5` as the best-performing regularization value.

The final Logistic Regression model is now trained on the complete training dataset so that all available observations are used before generating predictions for the test dataset.

In [10]:
# Create final Logistic Regression pipeline
model = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", LogisticRegression(
        max_iter=2000,
        C=best_C,
        solver="liblinear"
    ))
])

# Train on the complete training dataset
model.fit(X, y)

print("Final model trained successfully.")
print("Training observations used:", len(X))
print("Selected C:", best_C)

Final model trained successfully.
Training observations used: 9864
Selected C: 0.1


## 9. Create Processed Training Data

The fitted preprocessing pipeline is applied to the complete training feature set.

The resulting `processed_df` contains the transformed numerical and categorical features used by Logistic Regression.

No observations are removed during preprocessing, so 100% of the original training rows are retained.

In [11]:
# Get the fitted preprocessing component
fitted_preprocessor = model.named_steps["preprocessor"]

# Transform the complete training data
X_processed = fitted_preprocessor.transform(X)

# Get names of transformed features
processed_feature_names = fitted_preprocessor.get_feature_names_out()

# Create processed training dataframe
processed_df = pd.DataFrame(
    X_processed,
    columns=processed_feature_names,
    index=X.index
)

print("Original training shape :", original_df.shape)
print("Processed training shape:", processed_df.shape)

print("\nMissing values in processed_df:",
      processed_df.isnull().sum().sum())

print("\nRows retained:", len(processed_df))

print("Row retention:",
      round(len(processed_df) / len(original_df) * 100, 2),
      "%")

print("\nFirst 5 rows of processed data:")
display(processed_df.head())

Original training shape : (9864, 19)
Processed training shape: (9864, 35)

Missing values in processed_df: 0

Rows retained: 9864
Row retention: 100.0 %

First 5 rows of processed data:


,num__Administrative,num__Administrative_Duration,num__Informational,num__Informational_Duration,num__ProductRelated,num__ProductRelated_Duration,num__BounceRates,num__ExitRates,num__PageValues,num__SpecialDay,...,cat__VisitorType_Other,cat__VisitorType_Returning_Visitor,cat__Weekend_False,cat__Weekend_True,cat__PageValues_is_zero_0,cat__PageValues_is_zero_1,cat__BounceRates_is_zero_0,cat__BounceRates_is_zero_1,cat__ExitRates_is_zero_0,cat__ExitRates_is_zero_1
0,-0.919460,-1.005465,-0.476584,-0.464057,-1.329949,-1.049538,0.985912,1.899202,-0.492173,-0.315599,...,0.0,1.0,1.0,0.0,0.0,1.0,1.0,0.0,1.0,0.0
1,-0.919460,-1.005465,-0.476584,-0.464057,-0.282229,0.086831,0.867959,1.451168,-0.492173,-0.315599,...,0.0,1.0,1.0,0.0,0.0,1.0,1.0,0.0,1.0,0.0
2,0.942296,0.656378,-0.476584,-0.464057,-0.353380,-0.054457,-0.127020,0.265781,-0.492173,-0.315599,...,0.0,1.0,0.0,1.0,0.0,1.0,1.0,0.0,1.0,0.0
3,-0.919460,-1.005465,-0.476584,-0.464057,0.262766,0.143849,-0.471881,-0.718156,2.022715,-0.315599,...,0.0,0.0,1.0,0.0,1.0,0.0,0.0,1.0,1.0,0.0
4,1.485984,0.998170,-0.476584,-0.464057,0.144069,-0.180271,-0.222288,-0.050617,-0.492173,-0.315599,...,0.0,1.0,1.0,0.0,0.0,1.0,1.0,0.0,1.0,0.0


## 10. Generate Test Predictions

The final Logistic Regression model is applied to all observations in `test.csv`.

The preprocessing learned from the training data is automatically applied to the test data through the pipeline.

Predictions are generated for every `Session_ID`.

In [12]:
# Generate predictions for the complete test dataset
predictions = model.predict(X_test)

print("Number of test observations:", len(test_df))
print("Number of predictions:", len(predictions))

print("\nPrediction distribution:")
print(pd.Series(predictions).value_counts())

print("\nFirst 10 predictions:")
print(predictions[:10])

Number of test observations: 2466
Number of predictions: 2466

Prediction distribution:
0    2142
1     324
Name: count, dtype: int64

First 10 predictions:
[0 0 1 0 0 0 0 0 0 0]


## 11. Final Challenge Compliance Check

Before generating the submission, we verify that the solution satisfies all required competition conditions:

- Original training data is preserved.
- Processed data contains no missing values.
- At least 90% of training observations are retained.
- Logistic Regression is the final classification model.
- A prediction exists for every test observation.
- Every `Session_ID` is unique and represented exactly once.

In [13]:
print("===== FINAL CHALLENGE CHECK =====")

print("original_df shape:", original_df.shape)
print("processed_df shape:", processed_df.shape)
print("test_df shape:", test_df.shape)

print("\nOriginal missing values:",
      original_df.isnull().sum().sum())

print("Processed missing values:",
      processed_df.isnull().sum().sum())

print("\nOriginal rows:", len(original_df))
print("Processed rows:", len(processed_df))

row_retention = len(processed_df) / len(original_df) * 100
print(f"Row retention: {row_retention:.2f}%")

print("\nModel type:",
      model.named_steps["classifier"].__class__.__name__)

print("\nTest rows:", len(test_df))
print("Predictions:", len(predictions))

print("\nPrediction count matches test rows:",
      len(predictions) == len(test_df))

print("Session_IDs unique:",
      test_df["Session_ID"].nunique() == len(test_df))

print("\n===== CHECK COMPLETE =====")

===== FINAL CHALLENGE CHECK =====
original_df shape: (9864, 19)
processed_df shape: (9864, 35)
test_df shape: (2466, 18)

Original missing values: 2957
Processed missing values: 0

Original rows: 9864
Processed rows: 9864
Row retention: 100.00%

Model type: LogisticRegression

Test rows: 2466
Predictions: 2466

Prediction count matches test rows: True
Session_IDs unique: True

===== CHECK COMPLETE =====


In [15]:
import pandas as pd
import numpy as np

# ---------------------------------------------------
# Checkpoint information
# ---------------------------------------------------

original_missing = original_df.isnull().sum().sum()
processed_missing = processed_df.isnull().sum().sum()

original_rows, original_columns = original_df.shape
processed_rows, processed_columns = processed_df.shape

row_retained_percent = (
    processed_rows / original_rows
) * 100


# ---------------------------------------------------
# Detect model used
# ---------------------------------------------------

final_model = model

# Handle GridSearchCV / RandomizedSearchCV
if hasattr(final_model, "best_estimator_"):
    final_model = final_model.best_estimator_

# Handle sklearn Pipeline
if hasattr(final_model, "steps"):
    final_model = final_model.steps[-1][1]

model_name = final_model.__class__.__name__


# ---------------------------------------------------
# Create checkpoint section
# ---------------------------------------------------

checkpoints = pd.DataFrame({

    "id": [
        "original_missing",
        "processed_missing",
        "original_rows",
        "processed_rows",
        "original_columns",
        "processed_columns",
        "row_retained_percent",
        "model_name"
    ],

    "value": [
        original_missing,
        processed_missing,
        original_rows,
        processed_rows,
        original_columns,
        processed_columns,
        round(row_retained_percent, 2),
        model_name
    ]
})


# ---------------------------------------------------
# Create prediction section
# ---------------------------------------------------

prediction_output = pd.DataFrame({

    "id": test_df["Session_ID"].astype(str),

    "value": np.asarray(predictions).astype(str)

})


# ---------------------------------------------------
# Combine and save
# ---------------------------------------------------

submission = pd.concat(
    [checkpoints, prediction_output],
    ignore_index=True
)

submission.to_csv(
    "submission.csv",
    index=False
)

print("submission.csv created successfully.")
print(checkpoints)

submission.csv created successfully.
                     id               value
0      original_missing                2957
1     processed_missing                   0
2         original_rows                9864
3        processed_rows                9864
4      original_columns                  19
5     processed_columns                  35
6  row_retained_percent               100.0
7            model_name  LogisticRegression
